# ANALYSE DES STOCKS.

## I. Importation des donnees nettoyees.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [3]:
dossier_nettoye = "DATA_NETTOYEE"

os.makedirs(dossier_nettoye, exist_ok=True)
fichiers = os.listdir(dossier_nettoye)

print(fichiers)

['.gitkeep', 'achats_nettoye.csv', 'clients_nettoye.csv', 'commerciaux_nettoye.csv', 'depenses_nettoye.csv', 'fournisseurs_nettoye.csv', 'livraisons_nettoye.csv', 'magasins_nettoye.csv', 'produits_nettoye.csv', 'stocks_nettoye.csv', 'ventes_nettoye.csv']


In [4]:
ventes = pd.read_csv(os.path.join(dossier_nettoye, "ventes_nettoye.csv"))
clients = pd.read_csv(os.path.join(dossier_nettoye, "clients_nettoye.csv"))
achats = pd.read_csv(os.path.join(dossier_nettoye, "achats_nettoye.csv"))
produits = pd.read_csv(os.path.join(dossier_nettoye, "produits_nettoye.csv"))
commerciaux = pd.read_csv(os.path.join(dossier_nettoye, "commerciaux_nettoye.csv"))
fournisseurs = pd.read_csv(os.path.join(dossier_nettoye, "fournisseurs_nettoye.csv"))
magasins = pd.read_csv(os.path.join(dossier_nettoye, "magasins_nettoye.csv"))
depenses = pd.read_csv(os.path.join(dossier_nettoye, "depenses_nettoye.csv"))
livraisons = pd.read_csv(os.path.join(dossier_nettoye, "livraisons_nettoye.csv"))
stocks = pd.read_csv(os.path.join(dossier_nettoye, "stocks_nettoye.csv"))

## II. Analyses.

#### 1. Etat des stocks actuels.

In [5]:


# 1. Détail par type de mouvement (Ajustement, Entrée, Retour, Sortie)
mouvements = stocks.pivot_table(
    index='id_produit',
    columns='type_mouvement',
    values='quantite',
    aggfunc='sum',
    fill_value=0
).reset_index()

# 2. Somme réelle de tous les mouvements (les sorties étant déjà négatives)
mouvements['stock_disponible'] = stocks.groupby('id_produit')['quantite'].sum().values

# Fusionner avec le catalogue produits
df_stock = pd.merge(produits[['id_produit', 'nom_produit', 'categorie']], mouvements, on='id_produit', how='left').fillna(0)

# Classification du statut (Rupture <= 0, Surstock > 500 unités)
df_stock['statut_stock'] = df_stock['stock_disponible'].apply(
    lambda x: 'Rupture' if x <= 0 else ('Surstock' if x > 500 else 'Normal')
)

print("Répartition globale des stocks :")
print(df_stock['statut_stock'].value_counts())

print("\nTop 10 des produits en RUPTURE (stock le plus bas) :")
display(df_stock.sort_values(by='stock_disponible', ascending=True)[['nom_produit', 'categorie', 'Ajustement', 'Entrée', 'Retour', 'Sortie', 'stock_disponible', 'statut_stock']].head(10))

print("\nTop 10 des produits en SURSTOCK (quantité la plus élevée) :")
display(df_stock.sort_values(by='stock_disponible', ascending=False)[['nom_produit', 'categorie', 'Ajustement', 'Entrée', 'Retour', 'Sortie', 'stock_disponible', 'statut_stock']].head(10))

Répartition globale des stocks :
statut_stock
Rupture     197
Normal      161
Surstock    143
Name: count, dtype: int64

Top 10 des produits en RUPTURE (stock le plus bas) :


,nom_produit,categorie,Ajustement,Entrée,Retour,Sortie,stock_disponible,statut_stock
234,Biscuit Plus 235,Électronique,199.0,906.0,0.0,-2653.0,-1548.0,Rupture
401,Jus Nature 402,Entretien,56.0,1589.0,0.0,-3126.0,-1481.0,Rupture
489,Sucre Nature 490,Épicerie,178.0,796.0,18.0,-2160.0,-1168.0,Rupture
54,Café Plus 55,Entretien,72.0,649.0,0.0,-1865.0,-1144.0,Rupture
78,Huile Famille 79,Boisson,181.0,781.0,0.0,-2064.0,-1102.0,Rupture
247,Savon Eco 248,Produits Alimentaires,287.0,349.0,129.0,-1821.0,-1056.0,Rupture
70,Jus Famille 71,Produits Alimentaires,336.0,780.0,23.0,-2192.0,-1053.0,Rupture
46,Eau Famille 47,Produits Alimentaires,112.0,811.0,88.0,-2057.0,-1046.0,Rupture
390,Café Plus 391,Boisson,275.0,760.0,35.0,-2069.0,-999.0,Rupture
47,Sucre Nature 48,Entretien,234.0,998.0,0.0,-2231.0,-999.0,Rupture



Top 10 des produits en SURSTOCK (quantité la plus élevée) :


,nom_produit,categorie,Ajustement,Entrée,Retour,Sortie,stock_disponible,statut_stock
83,Eau Premium 84,Épicerie,459.0,2231.0,838.0,-1317.0,2211.0,Surstock
356,Eau Eco 357,Épicerie,272.0,2486.0,141.0,-895.0,2004.0,Surstock
313,Sucre Nature 314,Produits Alimentaires,276.0,1915.0,461.0,-1014.0,1638.0,Surstock
243,Sucre Premium 244,Entretien,367.0,2291.0,408.0,-1524.0,1542.0,Surstock
45,Détergent Classique 46,Boisson,155.0,1776.0,400.0,-790.0,1541.0,Surstock
104,Savon Plus 105,Électronique,398.0,1743.0,149.0,-830.0,1460.0,Surstock
17,Détergent Plus 18,Épicerie,247.0,2333.0,153.0,-1286.0,1447.0,Surstock
422,Jus Classique 423,Épicerie,543.0,1659.0,267.0,-1119.0,1350.0,Surstock
67,Farine Plus 68,Produits Alimentaires,422.0,1668.0,354.0,-1127.0,1317.0,Surstock
332,Lait Famille 333,Hygiène,434.0,1496.0,376.0,-1016.0,1290.0,Surstock


Explication métier des KPI calculés
Mouvements nets par type (Entrée, Sortie, Ajustement, Retour) : Mesurent l'ensemble des flux physiques entrants (réceptions fournisseurs, retours clients, régularisations positives) et sortants (ventes, casses, transferts).

Stock disponible (stock_disponible) : Quantité nette théorique présente en entrepôt (Entrées + Ajustements + Retours - Sorties). C'est le niveau réel disponible pour honorer les commandes clients.

Statut du stock (Rupture, Normal, Surstock) : Indicateur d'alerte opérationnelle permettant de segmenter le catalogue entre les produits indisponibles (<=0>), équilibrés (1 à 500 unités) et excessifs (> 500 unités).


Interprétation des résultats
Répartition déséquilibrée : Sur 501 références, 197 sont en rupture (39,3 %), 143 en surstock (28,5 %) et seulement 161 en niveau normal (32,1 %).

Anomalie de flux sur les ruptures : Les 10 pires ruptures affichent des soldes fortement négatifs (jusqu'à -1 548 unités sur Biscuit Plus 235). Cela révèle l'absence d'enregistrement du stock initial dans la base et/ou des oublis de saisie lors des réceptions de marchandises.

Accumulation sur les surstocks : À l'inverse, des produits comme Eau Premium 84 (+2 211 unités) ou Eau Eco 357 (+2 004 unités) accumulent des volumes massifs en raison d'achats non alignés sur le rythme réel des ventes.


Conclusion métier
67,8 % du catalogue est en situation critique (rupture ou surstock), traduisant un pilotage des approvisionnements déconnecté de la demande et des failles dans la traçabilité des entrées en entrepôt. Un inventaire physique général s'impose pour remettre les compteurs informatiques à zéro, accompagné d'un arrêt immédiat des commandes sur les 143 références en surstock.

#### 2. Performance des stocks.

In [6]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 1. Stock disponible par produit
stock_prod = stocks.groupby('id_produit')['quantite'].sum().reset_index().rename(columns={'quantite': 'stock_disponible'})

# 2. Ventes annuelles moyennes (en quantité) sur les ventes validées
ventes_valides = ventes[ventes['statut_vente'].str.lower().str.contains('valid', na=False)].copy()
ventes_valides['date_vente'] = pd.to_datetime(ventes_valides['date_vente'])
nb_jours_periode = (ventes_valides['date_vente'].max() - ventes_valides['date_vente'].min()).days
nb_annees = max(nb_jours_periode / 365.25, 1)

qte_vendue = ventes_valides.groupby('id_produit')['quantite'].sum().reset_index().rename(columns={'quantite': 'qte_vendue_totale'})
qte_vendue['ventes_annuelles_moy'] = qte_vendue['qte_vendue_totale'] / nb_annees

# 3. Fusion avec le catalogue produits
df_perf = produits[['id_produit', 'nom_produit', 'categorie', 'cout_achat_unitaire']].merge(stock_prod, on='id_produit', how='left').merge(qte_vendue, on='id_produit', how='left').fillna(0)

# 4. Calcul des indicateurs de performance (sur les produits ayant un stock positif)
df_perf['valeur_stock_fcfa'] = np.where(df_perf['stock_disponible'] > 0, df_perf['stock_disponible'] * df_perf['cout_achat_unitaire'], 0)
df_perf['rotation_stock'] = np.where(df_perf['stock_disponible'] > 0, df_perf['ventes_annuelles_moy'] / df_perf['stock_disponible'], np.nan)
df_perf['couverture_jours'] = np.where(df_perf['rotation_stock'] > 0, 365 / df_perf['rotation_stock'], np.nan)

# Nouvelle définition métier du surstock (> 90 jours de couverture)
df_perf['diagnostic_reel'] = np.where(
    df_perf['stock_disponible'] <= 0, 'Rupture',
    np.where(df_perf['couverture_jours'] > 90, 'Surstock (>90j)', 'Stock sain')
)

print(f"Valeur financière totale du stock positif immobilisé : {df_perf['valeur_stock_fcfa'].sum():,.0f} FCFA")
print("\nRépartition selon la couverture réelle en jours :")
print(df_perf['diagnostic_reel'].value_counts())

print("\nTop 10 des produits immobilisant le plus de trésorerie (Valeur financière) :")
display(df_perf.sort_values(by='valeur_stock_fcfa', ascending=False)[['nom_produit', 'categorie', 'stock_disponible', 'valeur_stock_fcfa', 'rotation_stock', 'couverture_jours', 'diagnostic_reel']].head(10))

print("\nTop 10 des produits à plus faible rotation (Durée de stockage la plus longue) :")
display(df_perf[df_perf['stock_disponible'] > 0].sort_values(by='couverture_jours', ascending=False)[['nom_produit', 'categorie', 'stock_disponible', 'ventes_annuelles_moy', 'rotation_stock', 'couverture_jours']].head(10))

Valeur financière totale du stock positif immobilisé : 1,915,323,501 FCFA

Répartition selon la couverture réelle en jours :
diagnostic_reel
Surstock (>90j)    299
Rupture            197
Stock sain           5
Name: count, dtype: int64

Top 10 des produits immobilisant le plus de trésorerie (Valeur financière) :


,nom_produit,categorie,stock_disponible,valeur_stock_fcfa,rotation_stock,couverture_jours,diagnostic_reel
304,Savon Nature 305,Électronique,1203.00,29515605.00,0.08,4663.76,Surstock (>90j)
0,Jus Classique 1,Produits Alimentaires,1231.00,28529656.00,0.05,6695.48,Surstock (>90j)
24,Biscuit Premium 25,Boisson,1119.00,25025316.00,0.06,5689.99,Surstock (>90j)
332,Lait Famille 333,Hygiène,1290.00,24729300.00,0.07,5552.34,Surstock (>90j)
179,Jus Plus 180,Hygiène,1118.00,23465702.00,0.06,6399.24,Surstock (>90j)
389,Jus Nature 390,Électronique,1025.00,23181400.00,0.08,4377.28,Surstock (>90j)
69,Sucre Plus 70,Épicerie,1275.00,22697550.00,0.06,6423.48,Surstock (>90j)
422,Jus Classique 423,Épicerie,1350.00,22628700.00,0.05,7607.68,Surstock (>90j)
104,Savon Plus 105,Électronique,1460.00,21593400.00,0.04,8917.02,Surstock (>90j)
363,Pâtes Nature 364,Entretien,1113.00,21221571.00,0.06,5794.23,Surstock (>90j)



Top 10 des produits à plus faible rotation (Durée de stockage la plus longue) :


,nom_produit,categorie,stock_disponible,ventes_annuelles_moy,rotation_stock,couverture_jours
356,Eau Eco 357,Épicerie,2004.00,59.76,0.03,12239.53
83,Eau Premium 84,Épicerie,2211.00,89.81,0.04,8985.79
104,Savon Plus 105,Électronique,1460.00,59.76,0.04,8917.02
17,Détergent Plus 18,Épicerie,1447.00,66.11,0.05,7989.57
422,Jus Classique 423,Épicerie,1350.00,64.77,0.05,7607.68
67,Farine Plus 68,Produits Alimentaires,1317.00,63.43,0.05,7577.96
313,Sucre Nature 314,Produits Alimentaires,1638.00,80.46,0.05,7430.48
314,Farine Plus 315,Boisson,981.00,50.75,0.05,7055.79
11,Détergent Plus 12,Hygiène,1080.00,56.76,0.05,6945.36
484,Eau Classique 485,Boisson,1141.00,60.43,0.05,6891.71


Explication des KPI
Valeur du stock immobilisé (valeur_stock_fcfa) : Capital financier bloqué dans l'entrepôt (Stock disponible * Coût d'achat unitaire).

Taux de rotation (rotation_stock) : Nombre de fois où le stock est renouvelé sur un an (Ventes annuelles moyennes / Stock disponible). Plus il est élevé, plus le produit s'écoule vite.

Couverture en jours (couverture_jours) : Nombre de jours de ventes assurés par le stock actuel (365 / Rotation). Au-delà de 90 jours, le produit est classé en surstock réel.


Interprétation
Trésorerie paralysée : 1,91 milliard FCFA sont immobilisés dans les stocks positifs, dont 29,5 M FCFA sur le seul Savon Nature 305.

Surstockage quasi généralisé : 299 produits (59,7 %) dépassent les 90 jours de couverture, contre seulement 5 produits (1 %) ayant un stock sain.

Rotation quasi nulle : Les produits dorment des années en entrepôt (rotation entre 0,03 et 0,08), comme Eau Eco 357 qui représente 12 239 jours (~33 ans) de ventes en stock.


Conclusion métier
L'entreprise immobilise près de 2 milliards FCFA dans des stocks à très faible rotation tout en subissant des ruptures sur près de 40 % de son catalogue. Il est urgent de liquider les surstocks (promotions, déstockage) pour libérer du cash et réallouer cette trésorerie vers le réapprovisionnement des produits en rupture.

#### 3. Fournisseurs & repprovisionnement.

In [7]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 1. Calcul des délais réels d'approvisionnement (sur les commandes reçues)
achats['date_commande'] = pd.to_datetime(achats['date_commande'])
achats['date_reception'] = pd.to_datetime(achats['date_reception'])
achats_recus = achats.dropna(subset=['date_reception']).copy()
achats_recus['delai_reel_jours'] = (achats_recus['date_reception'] - achats_recus['date_commande']).dt.days

# Fusion avec la table fournisseurs pour comparer au délai contractuel
df_fourn = achats_recus.merge(fournisseurs[['id_fournisseur', 'nom_fournisseur', 'type_fournisseur', 'delai_contractuel_jours']], on='id_fournisseur', how='left')
df_fourn['retard_jours'] = df_fourn['delai_reel_jours'] - df_fourn['delai_contractuel_jours']
df_fourn['en_retard'] = np.where(df_fourn['retard_jours'] > 0, 1, 0)

delai_moyen_global = df_fourn['delai_reel_jours'].mean()
taux_retard_global = df_fourn['en_retard'].mean() * 100

print(f"Délai moyen global d'approvisionnement : {delai_moyen_global:.1f} jours")
print(f"Taux global de commandes livrées en retard : {taux_retard_global:.1f} %")

# 2. Performance des fournisseurs (Top 10 pires retards moyens, avec au moins 10 commandes)
perf_fournisseurs = df_fourn.groupby(['id_fournisseur', 'nom_fournisseur', 'type_fournisseur']).agg(
    nb_commandes=('id_achat', 'count'),
    delai_contractuel=('delai_contractuel_jours', 'mean'),
    delai_reel_moy=('delai_reel_jours', 'mean'),
    retard_moyen=('retard_jours', 'mean'),
    taux_retard_pct=('en_retard', lambda x: x.mean() * 100)
).reset_index()

print("\nTop 10 des fournisseurs les moins performants (plus fort retard moyen) :")
display(perf_fournisseurs[perf_fournisseurs['nb_commandes'] >= 10].sort_values(by='retard_moyen', ascending=False).head(10))

# 3. Calcul du Stock Minimum, Point de Commande et Top 20 produits problématiques (sur df_perf de l'étape 2)
delai_par_produit = df_fourn.groupby('id_produit')['delai_reel_jours'].mean().reset_index().rename(columns={'delai_reel_jours': 'delai_appro_moy'})
df_opti = df_perf.merge(delai_par_produit, on='id_produit', how='left')
df_opti['delai_appro_moy'] = df_opti['delai_appro_moy'].fillna(delai_moyen_global)

# Ventes journalières moyennes
df_opti['ventes_jour_moy'] = df_opti['ventes_annuelles_moy'] / 365.0

# Stock minimum (sécurité = 15 jours de ventes) et Seuil de réapprovisionnement (Point de commande)
df_opti['stock_minimum_securite'] = np.ceil(df_opti['ventes_jour_moy'] * 15)
df_opti['seuil_reapprovisionnement'] = np.ceil((df_opti['ventes_jour_moy'] * df_opti['delai_appro_moy']) + df_opti['stock_minimum_securite'])

# Top 20 des produits les plus problématiques (10 pires ruptures + 10 plus gros surstocks financiers)
top10_ruptures = df_opti.sort_values(by='stock_disponible', ascending=True).head(10).copy()
top10_ruptures['probleme'] = 'Rupture critique'

top10_surstocks = df_opti.sort_values(by='valeur_stock_fcfa', ascending=False).head(10).copy()
top10_surstocks['probleme'] = 'Surstock financier majeur'

top20_problematiques = pd.concat([top10_ruptures, top10_surstocks], ignore_index=True)

print("\nTOP 20 DES PRODUITS LES PLUS PROBLÉMATIQUES & SEUILS RECOMMANDÉS :")
display(top20_problematiques[['probleme', 'nom_produit', 'categorie', 'stock_disponible', 'valeur_stock_fcfa', 'delai_appro_moy', 'stock_minimum_securite', 'seuil_reapprovisionnement']])

Délai moyen global d'approvisionnement : 18.0 jours
Taux global de commandes livrées en retard : 51.0 %

Top 10 des fournisseurs les moins performants (plus fort retard moyen) :


,id_fournisseur,nom_fournisseur,type_fournisseur,nb_commandes,delai_contractuel,delai_reel_moy,retard_moyen,taux_retard_pct
63,FOU064,Fournisseur 064 SARL,Grossiste,102,2.00,19.63,17.63,92.16
22,FOU023,Fournisseur 023 SARL,Grossiste,94,2.00,19.19,17.19,96.81
50,FOU051,Fournisseur 051 SA,International,94,2.00,18.84,16.84,96.81
21,FOU022,Fournisseur 022 SARL,International,96,4.00,18.52,14.52,90.62
87,FOU088,Fournisseur 088 SARL,Grossiste,97,3.00,17.38,14.38,88.66
99,FOU100,Fournisseur 100 SARL,Local,84,3.00,17.12,14.12,88.10
42,FOU043,Fournisseur 043 SARL,Grossiste,112,4.00,17.94,13.94,90.18
40,FOU041,Fournisseur 041 SARL,International,83,5.00,18.67,13.67,85.54
1,FOU002,Fournisseur 002 SARL,Local,95,5.00,18.62,13.62,88.42
59,FOU060,Fournisseur 060 SA,International,90,5.00,17.64,12.64,84.44



TOP 20 DES PRODUITS LES PLUS PROBLÉMATIQUES & SEUILS RECOMMANDÉS :


,probleme,nom_produit,categorie,stock_disponible,valeur_stock_fcfa,delai_appro_moy,stock_minimum_securite,seuil_reapprovisionnement
0,Rupture critique,Biscuit Plus 235,Électronique,-1548.00,0.00,22.93,5.00,12.00
1,Rupture critique,Jus Nature 402,Entretien,-1481.00,0.00,15.48,4.00,8.00
2,Rupture critique,Sucre Nature 490,Épicerie,-1168.00,0.00,16.44,3.00,7.00
3,Rupture critique,Café Plus 55,Entretien,-1144.00,0.00,15.69,3.00,6.00
4,Rupture critique,Huile Famille 79,Boisson,-1102.00,0.00,20.27,3.00,7.00
5,Rupture critique,Savon Eco 248,Produits Alimentaires,-1056.00,0.00,18.83,3.00,7.00
6,Rupture critique,Jus Famille 71,Produits Alimentaires,-1053.00,0.00,22.28,5.00,12.00
7,Rupture critique,Eau Famille 47,Produits Alimentaires,-1046.00,0.00,18.87,4.00,8.00
8,Rupture critique,Café Plus 391,Boisson,-999.00,0.00,19.67,4.00,10.00
9,Rupture critique,Sucre Nature 48,Entretien,-999.00,0.00,19.27,5.00,11.00


Explication des KPI
Délai réel d'approvisionnement (delai_reel_jours) : Temps écoulé entre la commande et la réception physique (Date réception - Date commande).

Taux et durée de retard (retard_jours, taux_retard_pct) : Écart entre le délai réel et le délai contractuel promis par le fournisseur, et pourcentage de livraisons hors délai.

Stock minimum de sécurité (stock_minimum_securite) : Réserve tampon (15 jours de ventes) pour absorber les retards fournisseurs ou pics de demande.

Point de commande (seuil_reapprovisionnement) : Niveau de stock déclenchant automatiquement une nouvelle commande (Ventes pendant le délai fournisseur + Stock minimum).


Interprétation
Fournisseurs peu fiables : Le délai moyen est de 18,0 jours, avec 51,0 % des commandes livrées en retard. Les pires fournisseurs (Fournisseur 064 SARL, Fournisseur 023 SARL, Fournisseur 051 SA) affichent plus de 92 % à 96 % de retards, livrant en ~19 jours au lieu des 2 jours contractuels.

Déconnexion des quantités commandées : Les seuils de réapprovisionnement optimaux (point de commande) se situent entre 6 et 12 unités selon la demande réelle, alors que l'entreprise commande et stocke plus de 1 000 à 1 400 unités sur des produits comme Savon Nature 305 ou Jus Classique 1.


Conclusion métier
La politique d'achat actuelle subit à la fois des retards fournisseurs chroniques (1 commande sur 2) et une absence de règles de commande (achats massifs au lieu de suivre la demande réelle). Il faut renégocier les contrats avec les fournisseurs défaillants (pénalités de retard, diversification) et automatiser les commandes sur la base des seuils de réapprovisionnement calculés (entre 6 et 12 unités).

In [11]:
ventes.columns

Index(['id_vente', 'date_vente', 'id_client', 'id_produit', 'id_magasin',
       'id_commercial', 'quantite', 'prix_unitaire_fcfa', 'remise_pct',
       'cout_unitaire_fcfa', 'mode_paiement', 'statut_vente', 'canal_vente',
       'chiffre_affaire_net', 'cout_total_ligne', 'marge_brute_ligne',
       'annee_mois'],
      dtype='str')